In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [4]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Record count: {df.count()}")
df.printSchema()

Record count: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
df.show(10, truncate=False)


+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [6]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp should now be 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(avg("amount"), 2).alias("avg_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+--------+----------+-------+
|   store|tx_count| total_PLN|avg_PLN|
+--------+--------+----------+-------+
|  Gdańsk|    2498|1021266.35| 408.83|
|  Kraków|    2522|1025896.95| 406.78|
|Warszawa|    2424| 961642.24| 396.72|
| Wrocław|    2556|1002739.21| 392.31|
+--------+--------+----------+-------+



In [9]:
#Count transactions, total revenue, min and max amount for each category.

from pyspark.sql.functions import min as _min, max as _max
category_stat = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(_min("amount"), 2).alias("min_txn"),
        _round(_max("amount"), 2).alias("max_txn"),
    )
    .orderBy("category")
)
category_stat.show()
# YOUR CODE
# df.groupBy("category").agg(...).orderBy("category").show()

+-----------+--------+----------+-------+-------+
|   category|tx_count| total_PLN|min_txn|max_txn|
+-----------+--------+----------+-------+-------+
|elektronika|    2542|1520770.69|    9.0| 9999.0|
|    książki|    2574| 851382.08|    5.0|9107.25|
|     odzież|    2453| 849877.55|    5.0|9696.63|
|    żywność|    2431| 789514.43|    5.0|6916.92|
+-----------+--------+----------+-------+-------+



In [11]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # 1-hour tumbling window
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+--------+----------+
|window                                    |tx_count|total_PLN |
+------------------------------------------+--------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150    |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661    |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189    |873403.24 |
+------------------------------------------+--------+----------+



In [12]:
(
    hourly
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150    |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189    |873403.24 |
+-------------------+-------------------+--------+----------+



In [13]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # 1h width, 30min step
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy("from")
)
sliding.show(truncate=False)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112    |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150    |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443    |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696    |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189    |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749     |289709.95 |
+-------------------+-------------------+--------+----------+



In [15]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):         {tumbling_rows} windows")
print(f"Sliding  (1h / 30min): {sliding_rows} windows")

# Answer in a comment: why does sliding produce more rows?
# Sliding windows produce more rows because they overlap.
# Each event can belong to multiple windows when the step size
# is smaller than the window duration, unlike tumbling windows
# where each event belongs to exactly one window.

Tumbling (1h):         3 windows
Sliding  (1h / 30min): 7 windows


In [16]:
###homework:
#1. Find the hour in which store Gdańsk had the lowest average transaction amount.
gdansk_hourly = (
    df.filter(col("store") == "Gdańsk")
      .groupBy(window("timestamp", "1 hour"))
      .agg(avg("amount").alias("avg_amount"))
      .orderBy("avg_amount")   # ascending → lowest first
)

lowest_hour = gdansk_hourly.select(
    col("window.start").alias("from"),
    col("window.end").alias("to"),
    "avg_amount"
).limit(1)

lowest_hour.show(truncate=False)

+-------------------+-------------------+-----------------+
|from               |to                 |avg_amount       |
+-------------------+-------------------+-----------------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.0118407310706|
+-------------------+-------------------+-----------------+



In [17]:
#2. Count how many transactions per category occurred in the 09:00–09:30 window.

filtered = df.filter(
    (col("timestamp") >= "2026-04-12 09:00:00") &
    (col("timestamp") <  "2026-04-12 09:30:00")
)

category_counts = (
    filtered.groupBy("category")
            .count()
            .orderBy("count", ascending=False)
)

category_counts.show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    książki|  622|
|elektronika|  611|
|     odzież|  605|
|    żywność|  567|
+-----------+-----+



In [18]:
#3.Use a 15-minute window and find which quarter-hour had the peak transaction volume (all stores combined).

quarter_hour = (
    df.groupBy(window("timestamp", "15 minutes"))
      .agg(count("tx_id").alias("tx_count"))
      .orderBy(col("tx_count").desc())
)

peak_window = quarter_hour.select(
    col("window.start").alias("from"),
    col("window.end").alias("to"),
    "tx_count"
).limit(1)

peak_window.show(truncate=False)

+-------------------+-------------------+--------+
|from               |to                 |tx_count|
+-------------------+-------------------+--------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234    |
+-------------------+-------------------+--------+

